In [4]:
##### Calculates final sub-national capital and labor intensities using final production and capital/labor rasters (after re-scaling)

from pathlib import Path
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from pyproj import Transformer
import matplotlib.patches as mpatches
from matplotlib.colors import to_hex, Normalize
from matplotlib import cm
import matplotlib
from matplotlib.patches import Rectangle
from rasterstats import zonal_stats

In [5]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# pixel data
capital = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD.tif")
labor = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs.tif")
production = rio.open_rasterio(f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif")

# sub-national data 
sub_national_capital_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_capital.shp")
sub_national_labor_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_labor.shp")

# savepath 
save_path_capital = f"{cd}/Results/Raster_model/sub_national_intensities/labor_intensity_subnational.csv"

In [6]:
### Data prep

target_crs = "+proj=eck4 +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"

capital = capital.rio.reproject(target_crs)

labor = labor.rio.reproject_match(capital)
production = production.rio.reproject_match(capital)

sub_national_capital_geo = sub_national_capital_geo.to_crs(capital.rio.crs)
sub_national_labor_geo = sub_national_labor_geo.to_crs(capital.rio.crs)

In [ ]:
##### Calculate sub-national capital and labor intensities (from model data aggregation)

def add_zonal_sum(gdf, raster_path, out_col, nodata):
    stats = zonal_stats(
        gdf,
        raster_path,
        stats=["sum"],
        nodata=nodata,
        all_touched=False  # set True if you want to include pixels only partially inside a polygon
    )
    gdf[out_col] = [s["sum"] for s in stats]
    return gdf

def compute_intensity(numerator_col, denominator_col):
    """
    Divide numerator by denominator, except:
    - if either numerator or denominator is 0 -> intensity is 0
    - if denominator is NaN (i.e. no valid pixels found in the zone at all) -> intensity is NaN
    """
    numerator = numerator_col
    denominator = denominator_col

    intensity = numerator / denominator  # normal division; produces inf/NaN in edge cases, overwritten below

    zero_mask = (numerator == 0) | (denominator == 0)
    intensity = intensity.where(~zero_mask, 0) if hasattr(intensity, "where") else np.where(zero_mask, 0, intensity)

    return intensity

# --- Capital intensity: sum capital and production within each capital sub-national region ---
sub_national_capital_geo = add_zonal_sum(
    sub_national_capital_geo,
    f"{cd}/Results/Raster_model/rescaled_capital_USD.tif",
    "capital_sum_USD",
    nodata=np.nan
)
sub_national_capital_geo = add_zonal_sum(
    sub_national_capital_geo,
    f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif",
    "production_sum_tonnes",
    nodata=-9999
)

sub_national_capital_geo["capital_intensity_USD_per_tonne"] = compute_intensity(
    sub_national_capital_geo["capital_sum_USD"],
    sub_national_capital_geo["production_sum_tonnes"]
)

# --- Labor intensity: sum labor and production within each labor sub-national region ---
sub_national_labor_geo = add_zonal_sum(
    sub_national_labor_geo,
    f"{cd}/Results/Raster_model/rescaled_jobs.tif",
    "labor_sum_jobs",
    nodata=np.nan
)
sub_national_labor_geo = add_zonal_sum(
    sub_national_labor_geo,
    f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif",
    "production_sum_tonnes",
    nodata=-9999
)

sub_national_labor_geo["labor_intensity_jobs_per_tonne"] = compute_intensity(
    sub_national_labor_geo["labor_sum_jobs"],
    sub_national_labor_geo["production_sum_tonnes"]
)